In [1]:
import pandas as pd
import numpy as np
from pptx import Presentation
from pptx.chart.data import CategoryChartData, XyChartData, BubbleChartData
from pptx.enum.chart import XL_CHART_TYPE, XL_TICK_MARK, XL_LEGEND_POSITION, XL_LABEL_POSITION, XL_MARKER_STYLE, XL_AXIS_CROSSES, XL_DATA_LABEL_POSITION
from pptx.enum.text import PP_ALIGN
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor 
import lxml.etree as ET
from pptx.oxml.xmlchemy import OxmlElement
from pptx.oxml.ns import qn

In [2]:
df = pd.read_pickle('scatter_data.pkl')

In [3]:
df

,spend,roi,contribution
media_idx,,,
FY'23 Media Digital Display,1.837482e+07,3.984991,801.474451
FY'23 Media Digital YT+Meta,5.007589e+07,2.152667,1179.903020
FY'23 Media OOH Retail,1.852609e+07,1.135843,230.131801
FY'23 Media OOH Screen,4.557492e+07,1.319789,658.462131
FY'23 Media Search Google,7.012253e+06,4.661931,357.717394
FY'24 Media Digital Display,0.000000e+00,0.000000,252.769224
FY'24 Media Digital YT+Meta,1.371983e+08,1.872105,2681.710762
FY'24 Media OOH Retail,3.217391e+07,2.877764,972.201564
FY'24 Media OOH Screen,0.000000e+00,0.000000,568.897799


In [4]:
prs = Presentation('./pptx_template/Kantar Master Template - LIFTROI ENG.pptx')
layout_blank = prs.slide_layouts[22]

In [9]:
_graph_slide = prs.slides.add_slide(layout_blank)
_graph_slide.placeholders[0].text = 'TEST'

_chart_data = XyChartData()
_series = _chart_data.add_series('Series')
for _, row in df.iterrows():
    _series.add_data_point(float(row['contribution']), float(row['roi'])) #, float(row['spend'])

_chart_type = XL_CHART_TYPE.XY_SCATTER
_graphic_frame = _graph_slide.shapes.add_chart(
    chart_type=_chart_type,
    x=Inches(0.39),
    y=Inches(1.87),
    cx=Inches(12.54),
    cy=Inches(4.37),
    chart_data=_chart_data
)

_chart = _graphic_frame.chart

_chart.has_title = False
_chart.has_legend = False

_x_axis = _chart.category_axis
_y_axis = _chart.value_axis

for _ax in (_x_axis, _y_axis):
    _ax.has_major_gridlines = False
    _ax.has_minor_gridlines = False
    _ax.major_tick_mark = XL_TICK_MARK.NONE
    _ax.minor_tick_mark = XL_TICK_MARK.NONE
    _ax.tick_labels.font.size = Pt(1)
    _ax.tick_labels.font.color.rgb = RGBColor(255, 255, 255)

if _chart_type == XL_CHART_TYPE.XY_SCATTER:
    _chart.series[0].marker.style = XL_MARKER_STYLE.CIRCLE
    _chart.series[0].marker.size = int(7)
    _chart.series[0].marker.format.fill.solid()
    _chart.series[0].marker.format.fill.fore_color.rgb = RGBColor(0,96,255)
    _chart.series[0].marker.format.line.fill.solid()
    _chart.series[0].marker.format.line.fill.fore_color.rgb = RGBColor(0,96,255)

if _chart_type == XL_CHART_TYPE.BUBBLE:
    _chart.series[0].format.fill.solid()
    _chart.series[0].format.fill.fore_color.rgb = RGBColor(0,96,255)
    _chart.series[0].format.line.fill.solid()
    _chart.series[0].format.line.fill.fore_color.rgb = RGBColor(0,96,255)

_xs = df['contribution'].astype(float)
_ys = df['roi'].astype(float)
def _pad_range(vals: pd.Series, ratio: float):
    vmin, vmax = vals.min(), vals.max()
    span = vmax - vmin
    if span == 0:
        span = max(abs(vmax), 1.0) * 0.1  # degenerate case
    pad = span * ratio
    return (vmin - pad, vmax + pad)

_x_min, _x_max = _pad_range(_xs, 0.05)
_y_min, _y_max = _pad_range(_ys, 0.05)

axis_center_mode = 'mean'
if axis_center_mode in ('zero', 'median', 'mean'):
    _x_center = 0.0 if axis_center_mode == 'zero' else (_xs.median() if axis_center_mode == 'median' else _xs.mean())
    _y_center = 0.0 if axis_center_mode == 'zero' else (_ys.median() if axis_center_mode == 'median' else _ys.mean())
    # Ensure center lies within scale
    _x_min = min(_x_min, _x_center)
    _x_max = max(_x_max, _x_center)
    _y_min = min(_y_min, _y_center)
    _y_max = max(_y_max, _y_center)


    # Where the Y axis crosses the X axis (on X scale)
    _x_axis.crosses = XL_AXIS_CROSSES.CUSTOM
    _x_axis.crosses_at = float(_x_center)
    # Where the X axis crosses the Y axis (on Y scale)
    _y_axis.crosses = XL_AXIS_CROSSES.CUSTOM
    _y_axis.crosses_at = float(_y_center)


_x_axis.minimum_scale = float(_x_min)
_x_axis.maximum_scale = float(_x_max)
_y_axis.minimum_scale = float(_y_min)
_y_axis.maximum_scale = float(_y_max)


# Enable data labels for the series
# _chart.plots[0].has_data_labels = True
# _chart.plots[0].data_labels.show_value = False  # Hide numeric values
# _chart.plots[0].data_labels.show_category_name = False
# _chart.plots[0].data_labels.show_series_name = False

# # Apply custom text from DataFrame index
# for i, point in enumerate(_chart.series[0].points):
#     point.data_label.text_frame.text = str(df.index[i])
#     # point.data_label.font.size = Pt(8)
#     # point.data_label.font.color.rgb = RGBColor(255, 255, 255)
#     point.data_label.position = XL_DATA_LABEL_POSITION.BEST_FIT


chart_xml = _chart.plots[0].chart._element
ser = chart_xml.find('.//c:ser', namespaces=chart_xml.nsmap)

# Ensure dLbls exists
dLbls = ser.find('c:dLbls', namespaces=chart_xml.nsmap)
if dLbls is None:
    dLbls = OxmlElement('c:dLbls')
    ser.append(dLbls)

for idx, label in enumerate(df.index):
    dLbl = OxmlElement('c:dLbl')
    
    idx_tag = OxmlElement('c:idx')
    idx_tag.set('val', str(idx))
    dLbl.append(idx_tag)

    tx = OxmlElement('c:tx')
    rich = OxmlElement('c:rich')
    bodyPr = OxmlElement('a:bodyPr')
    lstStyle = OxmlElement('a:lstStyle')
    p = OxmlElement('a:p')
    r = OxmlElement('a:r')
    rPr = OxmlElement('a:rPr')
    rPr.set('sz', '900')  # 9pt
    t = OxmlElement('a:t')
    t.text = str(label)

    r.append(rPr)
    r.append(t)
    p.append(r)
    rich.append(bodyPr)
    rich.append(lstStyle)
    rich.append(p)
    tx.append(rich)
    dLbl.append(tx)
    
    showLegendKey = OxmlElement('c:showLegendKey')
    showLegendKey.set('val', '0')
    dLbl.append(showLegendKey)


    dLbls.append(dLbl)


prs.save('test.pptx')